# Apache Beam Lab 2: Pipeline Fundamentals

## Overview
This lab introduces the fundamental concepts of Apache Beam pipelines. Understanding these core concepts is essential for building effective data processing pipelines.

## Learning Objectives
- Understand the Pipeline abstraction
- Learn about PCollections (Pipeline Collections)
- Master basic transforms: Create, Map, Filter, GroupByKey
- Understand pipeline execution model
- Practice building end-to-end pipelines

## Prerequisites
- Completed Lab 1: Environment Setup
- Sample data from Lab 0 available

## Core Concepts
### 1. Pipeline
The Pipeline is the main abstraction in Apache Beam. It encapsulates your entire data processing workflow.

In [ ]:
import apache_beam as beam

# Create a simple pipeline
def basic_pipeline():
    with beam.Pipeline() as pipeline:
        # Pipeline logic goes here
        pass

print("Pipeline concept: A Pipeline encapsulates your entire data processing workflow")

### 2. PCollection
A PCollection represents a distributed dataset that your pipeline will process. It's the fundamental data abstraction in Apache Beam.

In [ ]:
# Creating PCollections
def create_pcollections():
    with beam.Pipeline() as pipeline:
        # Create PCollection from in-memory data
        numbers = pipeline | 'Create numbers' >> beam.Create([1, 2, 3, 4, 5])
        
        # Create PCollection from text
        words = pipeline | 'Create words' >> beam.Create(['hello', 'world', 'beam'])
        
        # Create PCollection from sample data
        import pandas as pd
        sales_file = '/home/ramdov/projects/beam-code-practice/data/sample_sales.csv'
        if __import__('os').path.exists(sales_file):
            df = pd.read_csv(sales_file)
            data = df.to_dict('records')
            sales = pipeline | 'Create sales' >> beam.Create(data)

print("PCollection concept: A distributed dataset that your pipeline processes")

### 3. Transforms
Transforms are operations that process PCollections to produce new PCollections.

In [ ]:
# Basic transforms demonstration
def demonstrate_transforms():
    with beam.Pipeline() as pipeline:
        numbers = pipeline | 'Create' >> beam.Create([1, 2, 3, 4, 5])
        
        # Map transform: Apply function to each element
        doubled = numbers | 'Double' >> beam.Map(lambda x: x * 2)
        
        # Filter transform: Keep elements that meet condition
        evens = numbers | 'Filter even' >> beam.Filter(lambda x: x % 2 == 0)
        
        # Combine transform: Aggregate all elements
        total = numbers | 'Sum' >> beam.CombineGlobally(sum)
        
        # Print results
        doubled | 'Print doubled' >> beam.Map(lambda x: print(f"Doubled: {x}"))
        evens | 'Print evens' >> beam.Map(lambda x: print(f"Even: {x}"))
        total | 'Print total' >> beam.Map(lambda x: print(f"Total: {x}"))

print("Transforms: Operations that process PCollections to produce new PCollections")

## Exercise 1: Word Count Pipeline
Build a classic word count pipeline that processes text and counts word occurrences.

In [ ]:
# Your code here to build a word count pipeline
# 1. Create PCollection of text lines
# 2. Split lines into words
# 3. Normalize words (lowercase, remove punctuation)
# 4. Count occurrences of each word
# 5. Display results

def word_count_pipeline():
    """Build a word count pipeline."""
    import re
    
    with beam.Pipeline() as pipeline:
        (
            pipeline
            | 'Create lines' >> beam.Create([
                'Hello world',
                'Hello Apache Beam',
                'World of data processing',
                'Beam makes data processing easy'
            ])
            | 'Split into words' >> beam.FlatMap(lambda x: x.split())
            | 'Normalize' >> beam.Map(lambda x: re.sub(r'[^a-zA-Z]', '', x.lower()))
            | 'Filter empty' >> beam.Filter(lambda x: x)
            | 'Count words' >> beam.Map(lambda x: (x, 1))
            | 'Group and sum' >> beam.CombinePerKey(sum)
            | 'Format' >> beam.Map(lambda x: f"{x[0]}: {x[1]}")
            | 'Print' >> beam.Map(print)
        )

print("Running word count pipeline...")
word_count_pipeline()

## Exercise 2: Sales Analysis Pipeline
Build a pipeline that analyzes the sample sales data.

In [ ]:
# Your code here to build a sales analysis pipeline
# 1. Load sample sales data
# 2. Calculate total revenue
# 3. Find average transaction value
# 4. Count transactions by customer

import pandas as pd
import os

def sales_analysis_pipeline():
    """Analyze sales data with Apache Beam."""
    sales_file = '/home/ramdov/projects/beam-code-practice/data/sample_sales.csv'
    
    if not os.path.exists(sales_file):
        print("Sales data not found. Run Lab 0 first.")
        return
    
    df = pd.read_csv(sales_file)
    data = df.to_dict('records')
    
    with beam.Pipeline() as pipeline:
        # Calculate total revenue
        total_revenue = (
            pipeline
            | 'Create sales data' >> beam.Create(data)
            | 'Calculate transaction total' >> beam.Map(
                lambda x: x['quantity'] * x['price'])
            | 'Sum revenue' >> beam.CombineGlobally(sum)
        )
        
        # Calculate average transaction value
        avg_transaction = (
            pipeline
            | 'Create data for avg' >> beam.Create(data)
            | 'Calculate totals' >> beam.Map(
                lambda x: x['quantity'] * x['price'])
            | 'Calculate average' >> beam.CombineGlobally(
                lambda values: sum(values) / len(values) if values else 0)
        )
        
        # Count transactions by customer
        customer_counts = (
            pipeline
            | 'Create data for counts' >> beam.Create(data)
            | 'Extract customer' >> beam.Map(lambda x: (x['customer_id'], 1))
            | 'Count per customer' >> beam.CombinePerKey(sum)
        )
        
        # Print results
        total_revenue | 'Print revenue' >> beam.Map(
            lambda x: print(f"Total Revenue: ${x:.2f}"))
        avg_transaction | 'Print average' >> beam.Map(
            lambda x: print(f"Average Transaction: ${x:.2f}"))
        customer_counts | 'Print counts' >> beam.Map(
            lambda x: print(f"Customer {x[0]}: {x[1]} transactions"))

print("Running sales analysis pipeline...")
sales_analysis_pipeline()

## Exercise 3: Complex Pipeline with Multiple Branches
Build a pipeline that performs multiple analyses in parallel.

In [ ]:
# Your code here to build a complex pipeline with multiple branches
# 1. Load sample data once
# 2. Create multiple analysis branches from the same data
# 3. Perform different aggregations in parallel
# 4. Combine or display results

def complex_pipeline():
    """Build a complex pipeline with multiple analysis branches."""
    sales_file = '/home/ramdov/projects/beam-code-practice/data/sample_sales.csv'
    
    if not os.path.exists(sales_file):
        print("Sales data not found. Run Lab 0 first.")
        return
    
    df = pd.read_csv(sales_file)
    data = df.to_dict('records')
    
    with beam.Pipeline() as pipeline:
        # Create base PCollection
        sales_data = pipeline | 'Create sales data' >> beam.Create(data)
        
        # Branch 1: Revenue analysis
        (
            sales_data
            | 'Branch 1: Calculate revenue' >> beam.Map(
                lambda x: x['quantity'] * x['price'])
            | 'Branch 1: Sum revenue' >> beam.CombineGlobally(sum)
            | 'Branch 1: Print revenue' >> beam.Map(
                lambda x: print(f"Total Revenue: ${x:.2f}"))
        )
        
        # Branch 2: Product analysis
        (
            sales_data
            | 'Branch 2: Extract product' >> beam.Map(
                lambda x: (x['product_id'], x['quantity'] * x['price']))
            | 'Branch 2: Sum by product' >> beam.CombinePerKey(sum)
            | 'Branch 2: Format' >> beam.Map(
                lambda x: print(f"Product {x[0]}: ${x[1]:.2f}"))
        )
        
        # Branch 3: Customer analysis
        (
            sales_data
            | 'Branch 3: Extract customer' >> beam.Map(
                lambda x: (x['customer_id'], 1))
            | 'Branch 3: Count customers' >> beam.CombinePerKey(sum)
            | 'Branch 3: Format' >> beam.Map(
                lambda x: print(f"Customer {x[0]}: {x[1]} transactions"))
        )

print("Running complex pipeline with multiple branches...")
complex_pipeline()

## Summary

In this lab, you have:
- Understood the Pipeline abstraction
- Learned about PCollections and their properties
- Mastered basic transforms (Create, Map, Filter, Combine)
- Built word count and sales analysis pipelines
- Created complex pipelines with multiple branches

## Key Takeaways
- Pipelines encapsulate entire data processing workflows
- PCollections represent distributed datasets
- Transforms process PCollections to produce new PCollections
- Multiple analysis branches can operate on the same data
- Apache Beam provides a unified model for batch and streaming

## Next Steps

Proceed to [Lab 3: Core Transforms](lab-03-core-transforms.md) to learn about advanced transforms like ParDo and custom transformations.